# Problem Statement

The goal of this project is to predict whether a heart failure patient will be readmitted to the hospital within 30 days.

Hospital readmissions are a major challenge in healthcare because they increase costs and indicate poor patient outcomes.

We use machine learning to analyze both clinical and social factors such as:
- Vital signs (blood pressure, heart rate)
- Clinical markers (BNP, creatinine, sodium)
- Medication and treatment patterns
- Social factors (income level, access to care, medication adherence)

The objective is to build a predictive model that identifies high-risk patients early and helps reduce hospital readmissions.

#### Load dataset and import libbraries 

In [79]:
# Import Libraries for handling Data

import pandas as pd
import numpy as np

# Import train-test split
from sklearn.model_selection import train_test_split

# Import Logistic Regression
from sklearn.linear_model import LogisticRegression

# Import evaluation metrics
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Import SMOTE for handling class imbalance
from imblearn.over_sampling import SMOTE

# Import Random Forest model
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler

# Import SVM model
from sklearn.svm import SVC

# Import KNN model
from sklearn.neighbors import KNeighborsClassifier

# Import Naive Bayes model
from sklearn.naive_bayes import GaussianNB

In [7]:
# Load Datasets

df = pd.read_csv('heart_failure_readmission_dataset.csv')

# Data Head

df.head()


,patient_id,age,gender,bmi,bnp,sodium,creatinine,systolic_bp,heart_rate,ace_inhibitor,beta_blocker,diuretic,adherence_score,income_level,distance_to_hospital_km,readmitted_30d
0,12911,76,Male,23.9,738,135.3,1.58,151,93,1,1,0,0.98,Medium,12.4,0
1,12521,77,Male,32.3,405,143.0,1.50,107,74,1,0,1,0.66,Medium,38.8,1
2,10155,42,Male,29.3,399,NaN,1.43,121,97,1,0,1,0.93,Low,43.5,1
3,12088,83,Female,29.1,524,135.1,0.91,114,66,0,1,1,0.54,Low,33.3,1
4,10792,48,Female,24.2,301,139.5,0.54,122,79,1,1,1,0.78,High,21.3,0


In [13]:
# Check dataset shape (rows, columns)

print('Shape of Dataset:',df.shape)

# Check column names 
print("\nColumns:\n", df.columns)

# chedck for Missing values

print("\nMissing values:\n", df.isnull().sum())

# Basic infor about data types
print("\nData info:")
df.info()

Shape of Dataset: (3000, 16)

Columns:
 Index(['patient_id', 'age', 'gender', 'bmi', 'bnp', 'sodium', 'creatinine',
       'systolic_bp', 'heart_rate', 'ace_inhibitor', 'beta_blocker',
       'diuretic', 'adherence_score', 'income_level',
       'distance_to_hospital_km', 'readmitted_30d'],
      dtype='str')

Missing values:
 patient_id                  0
age                         0
gender                      0
bmi                        90
bnp                         0
sodium                     90
creatinine                 90
systolic_bp                 0
heart_rate                  0
ace_inhibitor               0
beta_blocker                0
diuretic                    0
adherence_score             0
income_level                0
distance_to_hospital_km     0
readmitted_30d              0
dtype: int64

Data info:
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------     

### Lets Handle missing values (ONLY missing values first)

We saw:

- bmi → 90 missing
- sodium → 90 missing
- creatinine → 90 missing

#### 🧠 Why median?

- Medical data has outliers
- Median is more stable than mean
- Prevents distortion of data


In [15]:
# Fill missing vlues with median (Safe for medical numerical data)

df['bmi'] =df['bmi'].fillna(df['bmi'].median())
df["sodium"] = df["sodium"].fillna(df["sodium"].median())
df["creatinine"] = df["creatinine"].fillna(df["creatinine"].median())

#### 🟢  Encode categorical columns

Now we convert text into numbers so the model can understand it.

In [19]:
# Convert gender into numbers

df['gender'] = df['gender'].map({"Male": 1, "Female": 0})

# Convert income level into numbers

df['income_level'] = df['income_level'].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

#### 🟢 Step 4.3: Final Data Check (Before Modeling)

In [ ]:
# Check first 5 rows
df.head()

,patient_id,age,gender,bmi,bnp,sodium,creatinine,systolic_bp,heart_rate,ace_inhibitor,beta_blocker,diuretic,adherence_score,income_level,distance_to_hospital_km,readmitted_30d
0,12911,76,1,23.9,738,135.3,1.58,151,93,1,1,0,0.98,1,12.4,0
1,12521,77,1,32.3,405,143.0,1.50,107,74,1,0,1,0.66,1,38.8,1
2,10155,42,1,29.3,399,138.1,1.43,121,97,1,0,1,0.93,0,43.5,1
3,12088,83,0,29.1,524,135.1,0.91,114,66,0,1,1,0.54,0,33.3,1
4,10792,48,0,24.2,301,139.5,0.54,122,79,1,1,1,0.78,2,21.3,0


In [21]:
# Check data type

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient_id               3000 non-null   int64  
 1   age                      3000 non-null   int64  
 2   gender                   3000 non-null   int64  
 3   bmi                      3000 non-null   float64
 4   bnp                      3000 non-null   int64  
 5   sodium                   3000 non-null   float64
 6   creatinine               3000 non-null   float64
 7   systolic_bp              3000 non-null   int64  
 8   heart_rate               3000 non-null   int64  
 9   ace_inhibitor            3000 non-null   int64  
 10  beta_blocker             3000 non-null   int64  
 11  diuretic                 3000 non-null   int64  
 12  adherence_score          3000 non-null   float64
 13  income_level             3000 non-null   int64  
 14  distance_to_hospital_km  3000 non-n

In [22]:
# Check if any missing values still exist

df.isnull().sum()

patient_id                 0
age                        0
gender                     0
bmi                        0
bnp                        0
sodium                     0
creatinine                 0
systolic_bp                0
heart_rate                 0
ace_inhibitor              0
beta_blocker               0
diuretic                   0
adherence_score            0
income_level               0
distance_to_hospital_km    0
readmitted_30d             0
dtype: int64

#### 🟢  Train/Test Split
Now we separate:

- features (X)
- target (y)

In [24]:
X= df.drop("readmitted_30d", axis=1)
y = df['readmitted_30d']

In [26]:
# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [27]:
# Check shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2400, 15)
X_test shape: (600, 15)
y_train shape: (2400,)
y_test shape: (600,)


#### 🟢  Logistic Regression (Baseline Model)

Now we build your first ML model.

In [29]:
# Create model

log_model = LogisticRegression(max_iter=1000)

# Train model

log_model.fit(X_train, y_train)

c:\Users\P15s\OneDrive\Documents\Logistic Regression Project\ml_env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

#### Make predictions

In [30]:
y_pred =log_model.predict(X_test)

#### Evaluate Logistic Regression Model

In [32]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[292  82]
 [128  98]]


In [35]:
# Classification Report
print('\nClassification Report:')
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.78      0.74       374
           1       0.54      0.43      0.48       226

    accuracy                           0.65       600
   macro avg       0.62      0.61      0.61       600
weighted avg       0.64      0.65      0.64       600



In [34]:
# Accuracy
print('\nAccuracy Score:')
print(accuracy_score(y_test, y_pred))


Accuracy Score:
0.65


##### 🟢 Model Evaluation Findings (Logistic Regression)

- Overall Accuracy = 65%
- The model is correct 65 out of 100 times
- This is moderate performance
- Not strong enough for healthcare use yet

##### 📊 2. Confusion Matrix Meaning

[[292  82]
 [128  98]]

 - 292 (True Negatives)
→ Correctly predicted patients who will NOT be readmitted
- 82 (False Positives)
→ Model wrongly predicted readmission (alarm but wrong)
- 128 (False Negatives) 🚨
→ Model FAILED to detect real readmitted patients
- 98 (True Positives)
→ Correctly detected readmitted patients

##### Most Important Problem

👉 The model is missing many sick patients

- Recall for class 1 = 0.43

##### This means:
- It only detects 43% of real readmissions
- It misses 57% of high-risk patients

##### Problem Class Imbalance Impact

- Class 0 (no readmission) = majority
- Class 1 (readmission) = minority

###### 👉 So the model learns:

“It is safer to predict 0 most of the time”

## Apply SMOTE (Balance Training Data Only)

In [37]:
# Create SMOTE object

smote = SMOTE(random_state=42)

# Apply SMOTE only on training data

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

#### Check New Class Balance

In [38]:
# Check the new balanced class distribution

print(y_train_smote.value_counts())

readmitted_30d
0    1392
1    1392
Name: count, dtype: int64


- D ataset is now fully balanced
- Both classes have equal importance
- The model will no longer “prefer” class 0
- This should improve recall for class 1 (readmitted patients)

##### Train Logistic Regression (SMOTE data)

In [39]:
# Create model

log_model_smote = LogisticRegression(max_iter=1000)

# Train using SMOTE data

log_model_smote.fit(X_train_smote, y_train_smote)


c:\Users\P15s\OneDrive\Documents\Logistic Regression Project\ml_env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

####  Predict on original test set

In [40]:
# Predict on original test set
y_pred_smote = log_model_smote.predict(X_test)

##### Evaluate SMOTE Logistic Regression Model

In [41]:
# Confusion Matrix

print('Confusion Matrix')
print(confusion_matrix(y_test, y_pred_smote))

Confusion Matrix
[[234 140]
 [ 90 136]]


In [42]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))


Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.63      0.67       374
           1       0.49      0.60      0.54       226

    accuracy                           0.62       600
   macro avg       0.61      0.61      0.61       600
weighted avg       0.64      0.62      0.62       600



In [43]:
# Accuracy Score
print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_smote))


Accuracy Score:
0.6166666666666667


##### 🟢 SMOTE Model Evaluation (Simple Explanation)

###### 📊 Confusion Matrix

[[234 140]
 [ 90 136]]
 
 Class 0 (Not readmitted)
- 234 correctly predicted
- 140 incorrectly predicted

👉 Model is still okay but slightly weaker than before

Class 1 (Readmitted) 🚨 IMPORTANT
- 136 correctly detected (GOOD ✔)
- 90 missed (still errors, but improved)

###### 📈 Key Improvements (VERY IMPORTANT)
Before SMOTE:
- Recall for class 1 = 0.43
- Model was missing most readmitted patients
After SMOTE:
- Recall for class 1 = 0.60 ✔

👉 That is a BIG improvement

###### 🧠 Final Interpretation

- Model now detects more high-risk patients
- Trade-off: slightly more false alarms
- Overall performance is more balanced

###### 📊 Accuracy = 61.6%

Lower than before (expected)
BUT more realistic and useful in healthcare

👉 In medical problems, recall is more important than accuracy



# Train Random Forest Model

We will now build a stronger model and compare it with Logistic Regression.

In [46]:
# Create model

rf_model = RandomForestClassifier(random_state=42)

# Train model on SMOTE data 

rf_model.fit(X_train_smote, y_train_smote)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [47]:
# Make prediction

y_pred_rf = rf_model.predict(X_test)

## Evaluate Random Forest (and compare with Logistic Regression 🚀)

In [48]:
# Confused Matrix
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))

Confusion Matrix
[[256 118]
 [ 96 130]]


In [49]:
# Classification Report

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.68      0.71       374
           1       0.52      0.58      0.55       226

    accuracy                           0.64       600
   macro avg       0.63      0.63      0.63       600
weighted avg       0.65      0.64      0.65       600



In [50]:
# Accuracy Score

print("\nAccuracy Score:")
print(accuracy_score(y_test,y_pred_rf))


Accuracy Score:
0.6433333333333333


### 🟢 Logistic Regression (SMOTE) vs Random Forest


| Metric              | Logistic Regression | Random Forest |
| ------------------- | ------------------- | ------------- |
| Accuracy            | 61.7%               | 64.3%         |
| Recall (Class 1)    | 0.60                | 0.58          |
| Precision (Class 1) | 0.49                | 0.52          |
| F1-score (Class 1)  | 0.54                | 0.55          |

### Accuracy improved

- Logistic Regression: 61.7%
- Random Forest: 64.3%

Random Forest makes more correct predictions overall.

### ✔ Precision improved

- Logistic Regression: 0.49
- Random Forest: 0.52

This means Random Forest produces fewer false alarms.

### ✔ F1-score improved

- Logistic Regression: 0.54
- Random Forest: 0.55

A small improvement in overall balance between precision and recall.



# Train XGBoost Model

In [57]:
# Create XGBoost model

xgb_model = XGBClassifier(random_state=42,eval_metric="logloss")

# Train model using SMOTE-balanced data
xgb_model.fit(X_train_smote, y_train_smote)


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [58]:
# Make predictions

y_pred_xgb =xgb_model.predict(X_test)

## Evaluate XGBoost Model

In [59]:
# Confuion Matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_xgb))

Confusion Matrix:
[[254 120]
 [111 115]]


In [60]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test,y_pred_xgb))


Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.68      0.69       374
           1       0.49      0.51      0.50       226

    accuracy                           0.61       600
   macro avg       0.59      0.59      0.59       600
weighted avg       0.62      0.61      0.62       600



In [61]:
# Accuracy Score

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_xgb))


Accuracy Score:
0.615


## 🚀 XGBoost (SMOTE)

- Accuracy: 61.5%
- Recall (Class 1): 0.51
Key Findings:
- XGBoost did not outperform the other models on this dataset.
- It shows moderate predictive ability.
- It misses more readmitted patients compared to other models.
- Requires further tuning to improve performance.

## 🧠 Overall Conclusion

- SMOTE significantly improved model performance on the minority class.
- There is a clear trade-off between accuracy and recall.
- Logistic Regression performed best for identifying high-risk patients.
- Random Forest performed best in overall balance.
- XGBoost was not the best fit for this dataset without tuning.


# Let Us improve more by Feature Scaling

In [65]:
scaler = StandardScaler()

# Fit only on training data

X_train_scaled = scaler.fit_transform(X_train_smote)

# Transform test data
X_test_scaled = scaler.transform(X_test)

## Support Vector Machine (SVM)

Because it is strong for classification problems.

In [68]:
# Create SVM model
svm_model = SVC()

# Train on scaled SMOTE data
svm_model.fit(X_train_scaled, y_train_smote)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [69]:
# Make predictions
y_pred_svm = svm_model.predict(X_test_scaled)

In [70]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_svm))

Confusion Matrix:
[[253 121]
 [ 85 141]]


In [71]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test,y_pred_svm))


Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.68      0.71       374
           1       0.54      0.62      0.58       226

    accuracy                           0.66       600
   macro avg       0.64      0.65      0.64       600
weighted avg       0.67      0.66      0.66       600



In [72]:
# Accuracy Score
print('\nAccuracy Score:')
print(accuracy_score(y_test, y_pred_svm))


Accuracy Score:
0.6566666666666666


#### 🟢 SVM Model Evaluation 
[[253 121]
 [ 85 141]]

### 🧠 What this means

✔ Class 0 (Not readmitted)
- 253 correct predictions
- 121 wrong predictions

👉 Model is fairly good at identifying stable patients

### Class 1 (Readmitted patients 🚨)

- 141 correctly detected ✔ (GOOD improvement)
- 85 missed ❌ (still some errors)

👉 This is important improvement in recall

## K-Nearest Neighbors (KNN)
KNN is a distance-based model, so it works well with your scaled data ✔



In [74]:
# Create KNN model
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train model
knn_model.fit(X_train_scaled, y_train_smote)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [75]:
# Make predictions
y_pred_knn = knn_model.predict(X_test_scaled)

In [76]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

Confusion Matrix:
[[241 133]
 [ 93 133]]


In [77]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))


Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.64      0.68       374
           1       0.50      0.59      0.54       226

    accuracy                           0.62       600
   macro avg       0.61      0.62      0.61       600
weighted avg       0.64      0.62      0.63       600



In [78]:
# Accuracy Score
print("\nAccurcy Score:")
print(accuracy_score(y_test, y_pred_knn))


Accurcy Score:
0.6233333333333333


## 🟢 KNN Model Evaluation

[[241 133]
 [ 93 133]]

 ### 🧠 What this means

✔ Class 0 (Not readmitted)
- 241 correct predictions
- 133 wrong predictions

👉 Model struggles a bit with stable patients


✔ Class 1 (Readmitted 🚨)
- 133 correctly detected ✔
- 93 missed ❌

👉 It catches moderate number of high-risk patients



| Model               | Accuracy | Recall (Class 1) | Comment             |
| ------------------- | -------- | ---------------- | ------------------- |
| Logistic Regression | 61%      | 0.60             | Good recall         |
| Random Forest       | 64%      | 0.58             | Balanced            |
| XGBoost             | 61%      | 0.51             | Weakest             |
| SVM                 | 66%      | 0.62 ⭐           | Best overall        |
| KNN                 | 62%      | 0.59             | Average performance |


# Let us See Naive Bayes Model



In [ ]:
# Create model
nb_model = GaussianNB()

# Train model (use SMOTE + scaled data? No scaling needed but OK to keep consistent)
nb_model.fit(X_train_smote, y_train_smote)

# Make predictions (use original test set, NOT scaled)
y_pred_nb = nb_model.predict(X_test)

In [81]:
# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Confusion Matrix:
[[247 127]
 [ 81 145]]


In [82]:

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))


Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.66      0.70       374
           1       0.53      0.64      0.58       226

    accuracy                           0.65       600
   macro avg       0.64      0.65      0.64       600
weighted avg       0.67      0.65      0.66       600



In [83]:
# Accuracy Score
print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_nb))


Accuracy Score:
0.6533333333333333


# Naive Bayes Evaluation

🧠 What this means

- ✔ Class 0 (Not readmitted)
- 247 correct predictions
- 127 wrong predictions

👉 Model is fairly stable for non-readmitted patients

- Class 1 (Readmitted 🚨)
- 145 correctly detected ✔ (GOOD)
- 81 missed ❌ (better than many models)

👉 This is actually a strong recall improvement

| Model               | Accuracy | Recall (Class 1) | Key Insight        |
| ------------------- | -------- | ---------------- | ------------------ |
| Logistic Regression | 61%      | 0.60             | Good recall        |
| Random Forest       | 64%      | 0.58             | Balanced           |
| XGBoost             | 61%      | 0.51             | Weak               |
| SVM                 | 66%      | 0.62             | Best overall model |
| KNN                 | 62%      | 0.59             | Average            |
| **Naive Bayes**     | **65%**  | **0.64 ⭐**       | Best recall so far |
